# CRISP-DM - Etapa 04: Modelagem e Agrupamento
## Tema 10: Indústria, Energia e Meio Ambiente (AI4I 2020 Predictive Maintenance)

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from sklearn.cluster import DBSCAN, KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score

sns.set_theme(style="whitegrid")

# carregar dados preparados
X = pd.read_parquet("../data/processed/ai4i2020_processed.parquet")
print("Dados preparados carregados:", X.shape)

### 1. Redução de Dimensionalidade com PCA (Visualização 2D)

In [ ]:
# projecao das variaveis em 2 componentes principais
pca = PCA(n_components=2)
X_pca = pca.fit_transform(X)

print(
    f"Variância explicada acumulado pelos 2 componentes: {sum(pca.explained_variance_ratio_):.2%}"
)

df_pca = pd.DataFrame(X_pca, columns=["PCA1", "PCA2"])
df_pca.head()

### 2. Agrupamento com K-Means (Método do Cotovelo e Silhueta)
Testando $k$ de 2 a 5 grupos (conforme restrição do negócio de no máximo 5 regimes)

In [ ]:
inercia = []
silhuetas = []
k_valores = range(2, 6)

for k in k_valores:
    kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = kmeans.fit_predict(X)
    inercia.append(kmeans.inertia_)
    silhuetas.append(silhouette_score(X, labels))

# plot de inercia e silhueta
fig, ax1 = plt.subplots(figsize=(10, 5))

ax1.plot(k_valores, inercia, "bo-", label="Inércia (Cotovelo)")
ax1.set_xlabel("Número de Clusters (k)")
ax1.set_ylabel("Inércia", color="b")

ax2 = ax1.twinx()
ax2.plot(k_valores, silhuetas, "ro--", label="Coeficiente de Silhueta")
ax2.set_ylabel("Silhueta", color="r")

plt.title("Avaliação do K-Means: Método do Cotovelo e Silhueta", fontsize=12, fontweight="bold")
plt.show()

### 3. Ajuste do K-Means Final (k = 3) e Visualização no Espaço PCA

In [ ]:
# treinamento do K-Means com k = 3
kmeans_final = KMeans(n_clusters=3, random_state=42, n_init=10)
df_pca["Cluster_KMeans"] = kmeans_final.fit_predict(X)

# plot dos clusters no plano 2D (PCA1 vs PCA2)
plt.figure(figsize=(10, 6))
sns.scatterplot(
    data=df_pca, x="PCA1", y="PCA2", hue="Cluster_KMeans", palette="Set1", s=30, alpha=0.7
)

plt.title("Clusters K-Means (k=3) Projetados no Espaço PCA", fontsize=12, fontweight="bold")
plt.show()

### 4. Experimento com Agrupamento Baseado em Densidade (DBSCAN)

In [ ]:
# aplicacao do DSBCAN para detectar outliers/regimes densos
dbscan = DBSCAN(eps=1.2, min_samples=15)
df_pca["Cluster_DBSCAN"] = dbscan.fit_predict(X)

print("Distribuição de pontos por cluster no DBSCAN (-1 indica ruído/outlier):")
print(df_pca["Cluster_DBSCAN"].value_counts())

# plot do DBSCAN no plano PCA
plt.figure(figsize=(10, 6))
sns.scatterplot(
    data=df_pca, x="PCA1", y="PCA2", hue="Cluster_DBSCAN", palette="tab10", s=30, alpha=0.7
)
plt.title("Clusters DBSCAN e Detecção de Outliers (-1)", fontsize=12, fontweight="bold")
plt.show()